# 01 — The Scaling Problem

## Question

Why does simply adding more quantum devices stop being an effective scaling strategy?

This notebook develops the engineering motivation behind integrated quantum photonics.

Roadmap

```
Need more channels
→ Replicate devices
→ Packaging complexity
→ Calibration burden
→ Compound yield
→ Multiplexing becomes preferable
``

In [ ]:
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import json
import zipfile
from IPython.display import FileLink

ROOT=Path.cwd()
BASE=ROOT.parent if ROOT.name=='notebooks' else ROOT

FIG=BASE/'figures'
RES=BASE/'results'
CSV=RES/'csv'
JS=RES/'json'

for p in [FIG,RES,CSV,JS]:
    p.mkdir(parents=True,exist_ok=True)

plt.rcParams['figure.dpi']=150
plt.rcParams['font.size']=12


## Figure 1 — Compound yield

Overall system yield decreases exponentially as independently fabricated components accumulate.

In [ ]:
eta=.99
N=np.arange(0,501)
yield_curve=eta**N

fig,ax=plt.subplots(figsize=(8,4.5))
ax.plot(N,yield_curve,lw=3)
ax.set_xlabel('Number of replicated devices')
ax.set_ylabel('Overall yield')
ax.set_title('Compound Yield')
ax.grid(alpha=.3)

for n in [10,100,300]:
    ax.scatter(n,eta**n,s=40)
    ax.text(n,eta**n+0.03,f'{n}')

fig.tight_layout()
fig.savefig(FIG/'01_compound_yield.png')
plt.show()

## Figure 2 — Packaging complexity

In [ ]:
devices=np.arange(1,17)
connections=devices**1.5

fig,ax=plt.subplots(figsize=(7,4.5))
ax.plot(devices,connections,'o-',lw=3)
ax.set_xlabel('Quantum devices')
ax.set_ylabel('Relative packaging complexity')
ax.set_title('Engineering Complexity Grows Faster Than Device Count')
ax.grid(alpha=.3)
fig.tight_layout()
fig.savefig(FIG/'01_packaging_growth.png')
plt.show()

## Figure 3 — Replication versus modal scaling

In [ ]:
generation=np.arange(1,17)
replication=generation
modal=np.array([1,2,3,5,8,12,17,24,32,45,60,75,90,110,140,170])

fig,ax=plt.subplots(figsize=(8,5))
ax.plot(generation,replication,'o-',label='Replication: one device → one channel',lw=3)
ax.plot(generation,modal,'s-',label='Modal scaling: richer optical modes',lw=3)
ax.set_xlabel('Engineering effort')
ax.set_ylabel('Available quantum channels')
ax.legend()
ax.grid(alpha=.3)
ax.set_title('Replication Versus Modal Scaling')
fig.tight_layout()
fig.savefig(FIG/'01_replication_vs_modes.png')
plt.show()

## Figure 4 — Scaling bottlenecks

In [ ]:
labels=['Fabrication','Packaging','Alignment','Calibration','Loss','Yield']
x=np.arange(len(labels))

fig,ax=plt.subplots(figsize=(11,2.8))

for i in range(len(labels)-1):
    ax.annotate('',xy=(i+1,0),xytext=(i,0),arrowprops=dict(arrowstyle='->',lw=2))

ax.scatter(x,np.zeros_like(x),s=350)

for xi,l in zip(x,labels):
    ax.text(xi,-.23,l,ha='center')

ax.axis('off')
ax.set_title('Engineering Bottlenecks')
fig.tight_layout()
fig.savefig(FIG/'01_scaling_bottlenecks.png')
plt.show()

## Figure 5 — Transition to multiplexing

In [ ]:
x=np.linspace(0,10,200)
rep=0.35*x
multi=.04*x**2

fig,ax=plt.subplots(figsize=(7,5))
ax.plot(x,rep,label='Replication cost',lw=3)
ax.plot(x,multi,label='Multiplexing capability',lw=3)
ax.fill_between(x,rep,multi,where=multi>rep,alpha=.2)
ax.set_xlabel('System scale')
ax.set_ylabel('Engineering return')
ax.legend()
ax.grid(alpha=.3)
ax.set_title('Transition Toward Multiplexing')
fig.tight_layout()
fig.savefig(FIG/'01_transition_to_multiplexing.png')
plt.show()

## Summary

In [ ]:
summary=pd.DataFrame({
'Constraint':['Replication','Packaging','Calibration','Yield','Transition'],
'Engineering effect':['More hardware','More interconnects','More tuning','Exponential decay','Multiplexing preferred'],
'Next notebook':['05_frequency_multiplexing']*5
})

summary.to_csv(CSV/'01_summary.csv',index=False)
summary.to_json(JS/'01_summary.json',orient='records',indent=2)

zip_path=RES/'01_outputs.zip'

with zipfile.ZipFile(zip_path,'w') as z:
    for f in FIG.glob('01_*.png'):
        z.write(f,f.name)
    z.write(CSV/'01_summary.csv','01_summary.csv')
    z.write(JS/'01_summary.json','01_summary.json')

print('Created:',zip_path)
display(FileLink(str(zip_path)))

summary